![Noteable.ac.uk Banner](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20NB%20Header%20Banner.png)

## Exemplar Information

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Purpose:</b> This exemplar introduces beginner-friendly bioinformatics and structural biology workflows for interpreting missense variants in a protein context. It shows how to move from a single amino-acid substitution to a careful molecular interpretation using sequence position, residue chemistry, domain context, and 3D structure. It covers data loading, simple feature engineering for amino-acid changes, sequence and domain mapping, structural distance calculations, visualisation with Plotly, interactive exploration with ipywidgets, and responsible scientific interpretation.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Intended audience / teaching context:</b> Designed for students in introductory bioinformatics, molecular bioscience, or beginner data analysis teaching sessions. Suitable for classroom demonstration, guided lab work, or independent practice in understanding how missense variants may affect protein function and structure.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Noteable Requirements:</b><br>
    <b>Environment:</b> LIVE 2026<br>
    <b>Server:</b> BioChemistry<br>
    <b>Kernel:</b> Python 3
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Required data or dependencies:</b>
    <br>
    - Curated teaching panel of <b>TP53</b> missense variants embedded directly in the notebook<br>
    - Optional internet access for fetching UniProt sequence and RCSB/PDB metadata<br>
    - Local or downloadable structural file for <b>1TUP</b> (TP53 complexed with DNA)<br>
    - Python packages: <code>numpy</code>, <code>pandas</code>, <code>plotly</code>, <code>ipywidgets</code>, <code>biopython</code>, <code>py3Dmol</code><br>
    - Standard library modules: <code>io</code>, <code>pathlib</code>, <code>json</code>, <code>urllib.request</code>
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Date created / last reviewed:</b> 13 July 2026<br>
    <b>Maintainer / owner:</b> Nik Yusuf
</div>


# From Variant to Function: Mapping Missense Changes onto Protein Context

## Legend

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>blue</b>, the <b>instructions</b> and <b>goals</b> are highlighted. This tells you what we are trying to achieve.
</div>
<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>green</b>, key <b>information</b> and <b>concept explanations</b> are highlighted.
</div>
<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>yellow</b>, <b>exercises</b> and <b>tasks</b> are highlighted for you to try yourself.
</div>
<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    In <b>red</b>, <b>error interpretation</b> and <b>debugging tips</b> are highlighted.
</div>

## 1. Why Missense Variants Matter

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Learn how to reason from an amino-acid substitution to a possible molecular consequence using sequence position, residue chemistry, domain context, and structural information.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    A <b>missense variant</b> changes one amino acid in a protein. Sometimes that change is quite subtle. Sometimes it alters charge, flexibility, packing, DNA contact, metal binding, or local stability. The key scientific skill is to separate <b>observation</b> from <b>interpretation</b> and from <b>speculation</b>.
</div>

In this notebook, we will use a compact teaching panel of well-known <b>TP53</b> missense changes. The p53 protein is a central regulator of cell stress responses, and many widely discussed variants sit in its DNA-binding domain.

Our aim is not to diagnose pathogenicity. Instead, we will ask a more careful biochemistry question:

**What kinds of molecular consequences become more plausible when we examine where a missense change happens and what sort of residue change it introduces?**

## 2. Setting Up Our Toolkit

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Import the tools we need for sequence handling, PDB access, structure parsing, interactive molecular viewing, and polished visual summaries.
</div>

Click the code cell below and press the <b><code>▶</code> Run</b> button in the toolbar at the top (or use <code>Shift + Enter</code>). The status messages will show you when each stage is ready.

In [ ]:
print("Starting setup: importing molecular bioscience libraries...")

from io import StringIO
from pathlib import Path
import json
import urllib.request

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML

from Bio import SeqIO
from Bio.PDB import PDBParser, PDBList  # FIXED: Added PDBList here
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.IUPACData import protein_letters_3to1
import py3Dmol

print("Success! Imports ready. We can now connect variants to protein context.")

## 3. Loading a Protein Context and a Curated Variant Panel

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Load a small, transparent teaching dataset of missense variants, then fetch a real p53 sequence and structure for context.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    This notebook uses a <b>manually curated teaching panel</b> of example missense variants. That makes the workflow light, reproducible, and easy to inspect. We are deliberately choosing clarity over scale.
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Try It:</b> The editable values in the next cell are safe to explore. You can swap the PDB ID, change the residue highlight cutoff later, or add your own example variants once the notebook is working.
</div>

In [ ]:
print("Loading the teaching variant panel and fetching protein context...")

# EDIT THIS VALUE: choose a protein/gene case study.
gene_name = "TP53"
protein_name = "Cellular tumour antigen p53"
uniprot_id = "P04637"

# EDIT THIS VALUE: choose a p53 DNA-binding-domain structure for structural context.
pdb_id = "1TUP"

# EDIT THIS VALUE: this variant panel is intentionally small and explained in class-friendly terms.
variant_records = [
    {"Variant": "P72R",  "Position": 72,  "WT": "P", "Mutant": "R", "Teaching focus": "Outside the core DNA-binding domain; useful comparison point."},
    {"Variant": "R175H", "Position": 175, "WT": "R", "Mutant": "H", "Teaching focus": "Classic structural hotspot close to the zinc-binding environment."},
    {"Variant": "Y220C", "Position": 220, "WT": "Y", "Mutant": "C", "Teaching focus": "Aromatic residue replaced by cysteine in the core domain."},
    {"Variant": "G245S", "Position": 245, "WT": "G", "Mutant": "S", "Teaching focus": "Glycine change in the DNA-binding domain; small but potentially important."},
    {"Variant": "R248Q", "Position": 248, "WT": "R", "Mutant": "Q", "Teaching focus": "A DNA-contact hotspot that removes a positive charge."},
    {"Variant": "R249S", "Position": 249, "WT": "R", "Mutant": "S", "Teaching focus": "Adjacent to a major DNA-contact residue."},
    {"Variant": "R273H", "Position": 273, "WT": "R", "Mutant": "H", "Teaching focus": "Another classic DNA-contact hotspot."},
    {"Variant": "R282W", "Position": 282, "WT": "R", "Mutant": "W", "Teaching focus": "Introduces a bulky aromatic residue in the DNA-binding domain."}
]

variant_df = pd.DataFrame(variant_records).sort_values("Position").reset_index(drop=True)

protein_length = 393
sequence_source = "Unavailable"
full_sequence = None

fasta_url = f"https://uniprot.org{uniprot_id}.fasta"
try:
    with urllib.request.urlopen(fasta_url) as response:
        fasta_text = response.read().decode("utf-8")
    fasta_record = SeqIO.read(StringIO(fasta_text), "fasta")
    full_sequence = str(fasta_record.seq)
    protein_length = len(full_sequence)
    sequence_source = f"UniProt {uniprot_id}"
except Exception as exc:
    print(f"Sequence download note (Environment Offline): Using manual fallbacks.")

# --- OFFLINE RESILIENT FILE DETECTION ---
data_dir = Path("variant_to_function_case_study")
data_dir.mkdir(exist_ok=True)

# Look for an already cached file from previous runs (e.g., pdb1tup.ent)
cached_files = list(data_dir.glob(f"*正式*{pdb_id.lower()}*")) + list(data_dir.glob(f"*{pdb_id.lower()}*"))
pdb_path = cached_files[0] if cached_files else None

if not pdb_path:
    try:
        pdbl = PDBList()
        fetched_file = pdbl.retrieve_pdb_file(pdb_id, pdir=str(data_dir), file_format="pdb")
        pdb_path = Path(fetched_file)
    except Exception as exc:
        print("Could not download structure file from network.")
        # Attempt standard naming fallback guessing
        pdb_path = data_dir / f"pdb{pdb_id.lower()}.ent"

# --- FIXED SECTION: OFFLINE-SAFE METADATA BLOCKS --- LC
structure_title = "Title not available"
experimental_method = "Not available"
resolution = np.nan

metadata_url = f"https://rcsb.org{pdb_id}"
try:
    with urllib.request.urlopen(metadata_url) as response:
        pdb_metadata = json.load(response)
    structure_title = pdb_metadata.get("struct", {}).get("title", structure_title)
    experimental_method = pdb_metadata.get("exptl", [{}])[0].get("method", experimental_method)
    resolution_list = pdb_metadata.get("rcsb_entry_info", {}).get("resolution_combined", [])
    resolution = resolution_list[0] if resolution_list else np.nan
except Exception:
    print("Metadata server unreachable. Using structural fallback information.")
    if pdb_id.upper() == "1TUP":
        structure_title = "TUMOR SUPPRESSOR P53 COMPLEXED WITH DNA"
        experimental_method = "X-RAY DIFFRACTION"
        resolution = 2.2
# ---------------------------------------------------

parser = PDBParser(QUIET=True)
if pdb_path and pdb_path.exists():
    structure = parser.get_structure(pdb_id, str(pdb_path))
    model = structure[0]
else:
    raise FileNotFoundError(f"Missing localized structural file {pdb_id} in {data_dir}. Please upload the .pdb or .ent file to continue offline.")

protein_chains = {}
for chain in model:
    aa_residues = [res for res in chain if is_aa(res, standard=True)]
    if aa_residues:
        protein_chains[chain.id] = aa_residues

selected_chain = max(protein_chains, key=lambda chain_id: len(protein_chains[chain_id]))
selected_chain_residues = protein_chains[selected_chain]
structure_residue_map = {res.id[1]: res for res in selected_chain_residues}
structure_positions = sorted(structure_residue_map.keys())

dna_resnames = {"DA", "DT", "DG", "DC", "A", "T", "G", "C", "U", "DU"}
dna_atoms = [atom for chain in model for residue in chain if residue.get_resname().strip() in dna_resnames for atom in residue.get_atoms() if atom.element != "H"]
zn_atoms = [atom for chain in model for residue in chain if residue.get_resname().strip() == "ZN" for atom in residue.get_atoms() if atom.element != "H"]

summary_df = pd.DataFrame(
    {
        "Property": [
            "Gene",
            "Protein",
            "Canonical sequence source",
            "Protein length used here",
            "PDB structure",
            "Chosen protein chain",
            "Structure title",
            "Experimental method",
            "Resolution (Å)",
            "Teaching variants"
        ],
        "Value": [
            gene_name,
            protein_name,
            sequence_source,
            protein_length,
            pdb_id,
            selected_chain,
            structure_title,
            experimental_method,
            round(float(resolution), 2) if np.isfinite(resolution) else "Not reported",
            len(variant_df)
        ]
    }
)

print("Variant data loaded. Protein context and structure are ready.")
display(summary_df)
display(variant_df)

## 4. Parsing the Amino-Acid Changes

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Turn each variant into interpretable biochemical features such as charge shift, polarity shift, size shift, and special-case residue changes.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    Not all missense changes are equally informative. A conservative-looking substitution can still matter in a sensitive structural position, while a dramatic-looking substitution may be tolerated in some contexts. That is why we combine residue chemistry with location.
</div>

In [ ]:
print("Parsing substitutions and adding biochemical and structural context...")

aa_properties = {
    "A": {"name": "Ala", "charge": "neutral", "polarity": "nonpolar", "size_rank": 1, "size_label": "small",  "aromatic": False},
    "R": {"name": "Arg", "charge": "positive", "polarity": "polar",    "size_rank": 4, "size_label": "large",  "aromatic": False},
    "N": {"name": "Asn", "charge": "neutral", "polarity": "polar",    "size_rank": 2, "size_label": "medium", "aromatic": False},
    "D": {"name": "Asp", "charge": "negative", "polarity": "polar",   "size_rank": 2, "size_label": "medium", "aromatic": False},
    "C": {"name": "Cys", "charge": "neutral", "polarity": "polar",    "size_rank": 2, "size_label": "medium", "aromatic": False},
    "Q": {"name": "Gln", "charge": "neutral", "polarity": "polar",    "size_rank": 3, "size_label": "large",  "aromatic": False},
    "E": {"name": "Glu", "charge": "negative", "polarity": "polar",   "size_rank": 3, "size_label": "large",  "aromatic": False},
    "G": {"name": "Gly", "charge": "neutral", "polarity": "nonpolar", "size_rank": 0, "size_label": "tiny",   "aromatic": False},
    "H": {"name": "His", "charge": "positive/neutral", "polarity": "polar", "size_rank": 3, "size_label": "large", "aromatic": True},
    "I": {"name": "Ile", "charge": "neutral", "polarity": "nonpolar", "size_rank": 3, "size_label": "large",  "aromatic": False},
    "L": {"name": "Leu", "charge": "neutral", "polarity": "nonpolar", "size_rank": 3, "size_label": "large",  "aromatic": False},
    "K": {"name": "Lys", "charge": "positive", "polarity": "polar",    "size_rank": 3, "size_label": "large",  "aromatic": False},
    "M": {"name": "Met", "charge": "neutral", "polarity": "nonpolar", "size_rank": 3, "size_label": "large",  "aromatic": False},
    "F": {"name": "Phe", "charge": "neutral", "polarity": "nonpolar", "size_rank": 4, "size_label": "large",  "aromatic": True},
    "P": {"name": "Pro", "charge": "neutral", "polarity": "nonpolar", "size_rank": 2, "size_label": "medium", "aromatic": False},
    "S": {"name": "Ser", "charge": "neutral", "polarity": "polar",    "size_rank": 1, "size_label": "small",  "aromatic": False},
    "T": {"name": "Thr", "charge": "neutral", "polarity": "polar",    "size_rank": 2, "size_label": "medium", "aromatic": False},
    "W": {"name": "Trp", "charge": "neutral", "polarity": "nonpolar", "size_rank": 4, "size_label": "large",  "aromatic": True},
    "Y": {"name": "Tyr", "charge": "neutral", "polarity": "polar",    "size_rank": 4, "size_label": "large",  "aromatic": True},
    "V": {"name": "Val", "charge": "neutral", "polarity": "nonpolar", "size_rank": 2, "size_label": "medium", "aromatic": False}
}

special_residues = {"G": "glycine", "P": "proline", "C": "cysteine"}

def domain_label(position):
    if 1 <= position <= 61:
        return "Transactivation region"
    if 62 <= position <= 93:
        return "Proline-rich region"
    if 94 <= position <= 292:
        return "DNA-binding domain"
    if 293 <= position <= 324:
        return "Linker / localisation region"
    if 325 <= position <= 356:
        return "Tetramerisation domain"
    return "Regulatory tail"

def min_distance(residue, atoms):
    if residue is None or len(atoms) == 0:
        return np.nan
    residue_atoms = [atom for atom in residue.get_atoms() if atom.element != "H"]
    if len(residue_atoms) == 0:
        return np.nan
    return float(min(np.linalg.norm(atom.coord - other.coord) for atom in residue_atoms for other in atoms))

def structure_one_letter(position):
    residue = structure_residue_map.get(position)
    if residue is None:
        return None
    return protein_letters_3to1.get(residue.get_resname().title())

def local_sequence_window(position, window=6):
    if full_sequence is None or position < 1 or position > len(full_sequence):
        return None
    start = max(1, position - window)
    end = min(len(full_sequence), position + window)
    subseq = full_sequence[start - 1:end]
    highlighted = "".join(
        f"<span style='color:#b30000; font-weight:bold; background:#fff3cd; padding:1px 3px; border-radius:3px;'>{aa}</span>" if (start + idx) == position else aa
        for idx, aa in enumerate(subseq)
    )
    return start, end, highlighted

processed_rows = []
for _, row in variant_df.iterrows():
    wt = row["WT"]
    mut = row["Mutant"]
    position = int(row["Position"])
    wt_props = aa_properties[wt]
    mut_props = aa_properties[mut]

    sequence_wt = full_sequence[position - 1] if full_sequence is not None and position <= len(full_sequence) else None
    structure_wt = structure_one_letter(position)
    residue_obj = structure_residue_map.get(position)

    charge_changed = wt_props["charge"] != mut_props["charge"]
    polarity_changed = wt_props["polarity"] != mut_props["polarity"]
    size_delta = mut_props["size_rank"] - wt_props["size_rank"]
    aromatic_changed = wt_props["aromatic"] != mut_props["aromatic"]

    special_notes = []
    if wt in special_residues:
        special_notes.append(f"removes {special_residues[wt]}")
    if mut in special_residues:
        special_notes.append(f"introduces {special_residues[mut]}")
    special_case = "; ".join(special_notes) if special_notes else "None"

    conservative = (
        wt_props["charge"] == mut_props["charge"]
        and wt_props["polarity"] == mut_props["polarity"]
        and abs(size_delta) <= 1
        and wt_props["aromatic"] == mut_props["aromatic"]
        and wt not in special_residues
        and mut not in special_residues
    )

    biochemical_shift_score = int(charge_changed) + int(polarity_changed) + int(abs(size_delta) > 1) + int(aromatic_changed) + int(special_case != "None")

    dna_distance = min_distance(residue_obj, dna_atoms)
    zn_distance = min_distance(residue_obj, zn_atoms)
    near_dna = bool(np.isfinite(dna_distance) and dna_distance < 6.0)
    near_zn = bool(np.isfinite(zn_distance) and zn_distance < 6.0)

    context_score = biochemical_shift_score + int(domain_label(position) == "DNA-binding domain") + int(near_dna) + int(near_zn)

    processed_rows.append(
        {
            "Variant": row["Variant"],
            "Position": position,
            "WT": wt,
            "Mutant": mut,
            "Teaching focus": row["Teaching focus"],
            "Domain": domain_label(position),
            "Sequence WT check": sequence_wt,
            "Sequence matches expected": (sequence_wt == wt) if sequence_wt is not None else "Sequence unavailable",
            "Structure WT check": structure_wt,
            "In chosen structure?": position in structure_residue_map,
            "Charge change": f"{wt_props['charge']} → {mut_props['charge']}" if charge_changed else "No charge change",
            "Polarity change": f"{wt_props['polarity']} → {mut_props['polarity']}" if polarity_changed else "No polarity change",
            "Size change": f"{wt_props['size_label']} → {mut_props['size_label']}",
            "Aromaticity change": "Yes" if aromatic_changed else "No",
            "Special-case note": special_case,
            "Change class": "Conservative-looking" if conservative else "Non-conservative",
            "Biochemical shift score": biochemical_shift_score,
            "DNA distance (Å)": round(dna_distance, 2) if np.isfinite(dna_distance) else np.nan,
            "Zinc distance (Å)": round(zn_distance, 2) if np.isfinite(zn_distance) else np.nan,
            "Near DNA in structure?": near_dna,
            "Near zinc in structure?": near_zn,
            "Teaching context score": context_score
        }
    )

variant_df = pd.DataFrame(processed_rows).sort_values("Position").reset_index(drop=True)

def short_interpretation(row):
    ideas = []
    ideas.append(f"This variant sits in the {row['Domain'].lower()}.")
    ideas.append("The residue change looks conservative at first glance." if row["Change class"] == "Conservative-looking" else "The residue change is biochemically non-conservative.")
    if row["Charge change"] != "No charge change":
        ideas.append(f"It changes charge class: {row['Charge change'].lower()}.")
    if row["Special-case note"] != "None":
        ideas.append(f"It also {row['Special-case note']}.")
    if row["Near DNA in structure?"]:
        ideas.append("The mapped position lies close to DNA in the chosen structure, so a direct DNA-contact effect becomes more plausible.")
    if row["Near zinc in structure?"]:
        ideas.append("The mapped position is also close to the zinc-binding environment, which makes local structural stability especially relevant.")
    if not row["In chosen structure?"]:
        ideas.append("This position is outside the selected structure, so our interpretation here relies more on sequence and domain context than on direct structural mapping.")
    ideas.append("This is still an interpretation aid, not proof of mechanism.")
    return " ".join(ideas)

variant_df["Why this might matter"] = variant_df.apply(short_interpretation, axis=1)

print("Substitutions parsed successfully.")
display(variant_df[[
    "Variant", "Domain", "Change class", "Charge change", "Polarity change", "Size change",
    "Special-case note", "DNA distance (Å)", "Zinc distance (Å)", "Teaching context score"
]])

## 5. Where Do the Variants Sit Along the Protein?

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> Visualise where the variants fall across the full p53 sequence and compare their simple biochemical-shift scores.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    The score below is a <b>teaching heuristic</b>. It is not a pathogenicity score. It simply counts how many residue-property changes are happening at once, then lets us compare that with domain and structural context.
</div>

In [ ]:
print("Building a sequence-position map for the curated missense variants...")

domain_blocks = [
    (1, 61, "Transactivation region", "rgba(141,211,199,0.20)"),
    (62, 93, "Proline-rich region", "rgba(255,255,179,0.25)"),
    (94, 292, "DNA-binding domain", "rgba(251,128,114,0.18)"),
    (293, 324, "Linker / localisation region", "rgba(190,186,218,0.20)"),
    (325, 356, "Tetramerisation domain", "rgba(128,177,211,0.20)"),
    (357, protein_length, "Regulatory tail", "rgba(253,180,98,0.20)")
]

fig = go.Figure()

for start, end, label, color in domain_blocks:
    fig.add_vrect(x0=start, x1=end, fillcolor=color, opacity=1.0, line_width=0, annotation_text=label, annotation_position="top left")

for _, row in variant_df.iterrows():
    marker_color = "#C44E52" if row["Change class"] == "Non-conservative" else "#4C72B0"
    marker_symbol = "diamond" if row["In chosen structure?"] else "circle"
    fig.add_trace(
        go.Scatter(
            x=[row["Position"], row["Position"]],
            y=[0, row["Biochemical shift score"]],
            mode="lines",
            line=dict(color="rgba(80,80,80,0.45)", width=2),
            hoverinfo="skip",
            showlegend=False
        )
    )
    fig.add_trace(
        go.Scatter(
            x=[row["Position"]],
            y=[row["Biochemical shift score"]],
            mode="markers+text",
            text=[row["Variant"]],
            textposition="top center",
            marker=dict(size=14, color=marker_color, symbol=marker_symbol, line=dict(color="black", width=0.5)),
            name=row["Change class"],
            hovertemplate=(
                f"<b>{row['Variant']}</b><br>"
                f"Position: {row['Position']}<br>"
                f"Domain: {row['Domain']}<br>"
                f"Change class: {row['Change class']}<br>"
                f"Biochemical shift score: {row['Biochemical shift score']}<br>"
                f"Teaching focus: {row['Teaching focus']}<extra></extra>"
            ),
            showlegend=False
        )
    )

fig.update_layout(
    template="plotly_white",
    title="Sequence-position map of curated TP53 missense variants",
    xaxis_title="Residue position",
    yaxis_title="Biochemical shift score (teaching heuristic)",
    height=520
)
fig.update_xaxes(range=[1, protein_length + 5])
fig.update_yaxes(range=[0, max(variant_df['Biochemical shift score']) + 2])
fig.show()

print("Sequence mapping complete. Notice how many of these example variants cluster in the DNA-binding domain.")

In [ ]:
print("Summarising substitution classes and structural context...")

summary_counts = pd.DataFrame(
    {
        "Category": [
            "Conservative-looking",
            "Non-conservative",
            "Charge-changing",
            "Special-case residue involved",
            "Mapped near DNA (< 6 Å)",
            "Mapped near zinc (< 6 Å)"
        ],
        "Count": [
            int((variant_df["Change class"] == "Conservative-looking").sum()),
            int((variant_df["Change class"] == "Non-conservative").sum()),
            int((variant_df["Charge change"] != "No charge change").sum()),
            int((variant_df["Special-case note"] != "None").sum()),
            int(variant_df["Near DNA in structure?"].sum()),
            int(variant_df["Near zinc in structure?"].sum())
        ]
    }
)

bar_fig = px.bar(
    summary_counts,
    x="Category",
    y="Count",
    color="Count",
    color_continuous_scale="Tealgrn",
    title="What kinds of changes are represented in this teaching panel?"
)
bar_fig.update_layout(
    template="plotly_white",
    xaxis_title="",
    yaxis_title="Number of variants",
    coloraxis_showscale=False
)
bar_fig.show()

structured_subset = variant_df[variant_df["In chosen structure?"]].copy()
if not structured_subset.empty:
    scatter_fig = px.scatter(
        structured_subset,
        x="DNA distance (Å)",
        y="Zinc distance (Å)",
        size="Teaching context score",
        color="Change class",
        hover_name="Variant",
        hover_data=["Domain", "Teaching focus"],
        title="Structural context of mapped variants in the chosen p53 structure"
    )
    scatter_fig.add_vline(x=6, line_dash="dash", line_color="gray")
    scatter_fig.add_hline(y=6, line_dash="dash", line_color="gray")
    scatter_fig.update_layout(template="plotly_white")
    scatter_fig.show()

display(structured_subset[["Variant", "DNA distance (Å)", "Zinc distance (Å)", "Teaching context score"]].sort_values("Teaching context score", ascending=False))

print("Visual summaries ready. We can now compare residue chemistry with structural neighbourhood.")

## 6. Adding 3D Structural Context

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Goal:</b> See where the curated variant positions sit in a real p53 structure alongside DNA and the zinc ion.
</div>

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    Structural mapping helps us ask better questions. Is the residue close to DNA? Near zinc? Buried in the core? On a loop? A 3D view does not answer everything, but it makes our interpretation much more concrete.
</div>

In [ ]:
print("Rendering a 3D overview of the structure and mapped variant sites...")

# 1. FIXED: Read the local cached PDB file from disk into a string variable
with open(pdb_path, "r") as f:
    pdb_text = f.read()

mapped_positions = variant_df.loc[variant_df["In chosen structure?"], "Position"].astype(str).tolist()
mapped_selection = ",".join(mapped_positions)

view = py3Dmol.view(width=950, height=560)
# 2. Uses the newly defined local pdb_text string variable safely
view.addModel(pdb_text, "pdb")
view.setStyle({"chain": selected_chain}, {"cartoon": {"color": "lightgray"}})
view.addStyle({"resn": ["A", "T", "G", "C", "DA", "DT", "DG", "DC"]}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.18}})
view.addStyle({"resn": "ZN"}, {"sphere": {"color": "green", "scale": 0.9}})

if mapped_selection:
    view.addStyle({"chain": selected_chain, "resi": mapped_selection}, {"stick": {"colorscheme": "redCarbon", "radius": 0.22}})
    view.addStyle({"chain": selected_chain, "resi": mapped_selection}, {"sphere": {"color": "red", "scale": 0.28}})

view.zoomTo({"chain": selected_chain})
view.setBackgroundColor("white")
display(HTML(view._make_html()))

print("3D structure ready. DNA is orange, zinc is green, and mapped teaching variants are highlighted in red.")

## 7. Interactive Variant Explorer

<div style="border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Interactivity:</b> Use the controls below to choose a variant and inspect its residue chemistry, sequence neighbourhood, and 3D mapping all together.
</div>

<div style="border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Try It:</b> Compare <b>R248Q</b> with <b>P72R</b>, then compare <b>R175H</b> with <b>Y220C</b>. Ask yourself: which differences come from residue chemistry, and which come from structural position?
</div>

In [ ]:
print("Building the interactive variant explorer...")

variant_lookup = variant_df.set_index("Variant")

def explore_variant(selected_variant, window_size=6, show_structure=True):
    row = variant_lookup.loc[selected_variant]

    summary = pd.DataFrame(
        {
            "Property": [
                "Variant",
                "Domain",
                "Teaching focus",
                "Change class",
                "Charge change",
                "Polarity change",
                "Size change",
                "Special-case note",
                "DNA distance (Å)",
                "Zinc distance (Å)",
                "Teaching context score"
            ],
            "Value": [
                selected_variant,
                row["Domain"],
                row["Teaching focus"],
                row["Change class"],
                row["Charge change"],
                row["Polarity change"],
                row["Size change"],
                row["Special-case note"],
                row["DNA distance (Å)"],
                row["Zinc distance (Å)"],
                row["Teaching context score"]
            ]
        }
    )
    display(summary)

    interpretation_html = f"""
    <div style='border:1px solid #c3e6cb; background:#d4edda; color:#155724; padding:14px; margin:8px 0; border-radius:4px;'>
        <b>Why this substitution might matter:</b><br>{row['Why this might matter']}
    </div>
    """
    display(HTML(interpretation_html))

    window_info = local_sequence_window(int(row["Position"]), window=window_size)
    if window_info is not None:
        start, end, highlighted = window_info
        sequence_html = f"""
        <div style='font-family:monospace; border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:12px; margin:8px 0; border-radius:4px;'>
            <b>Local sequence window</b> (positions {start}-{end})<br>{highlighted}
        </div>
        """
        display(HTML(sequence_html))
    else:
        display(HTML("<div style='border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:12px; margin:8px 0; border-radius:4px;'><b>Sequence note:</b> Full sequence context was not downloaded, so the local sequence window is unavailable in this session.</div>"))

    if show_structure:
        in_structure = bool(row["In chosen structure?"])
        variant_pos = int(row["Position"])
        
        local_view = py3Dmol.view(width=900, height=520)
        local_view.addModel(pdb_text, "pdb")
        local_view.setStyle({"chain": selected_chain}, {"cartoon": {"color": "lightgray"}})
        local_view.addStyle({"resn": ["A", "T", "G", "C", "DA", "DT", "DG", "DC"]}, {"stick": {"colorscheme": "orangeCarbon", "radius": 0.18}})
        local_view.addStyle({"resn": "ZN"}, {"sphere": {"color": "green", "scale": 0.9}})
        
        if in_structure:
            # Highlight the variant normally if it exists in the PDB file
            local_view.addStyle({"chain": selected_chain, "resi": str(variant_pos)}, {"stick": {"colorscheme": "redCarbon", "radius": 0.28}})
            local_view.addStyle({"chain": selected_chain, "resi": str(variant_pos)}, {"sphere": {"color": "red", "scale": 0.35}})
            local_view.addLabel(selected_variant, {"fontColor": "black", "backgroundColor": "white", "showBackground": True}, {"chain": selected_chain, "resi": str(variant_pos)})
            local_view.zoomTo({"chain": selected_chain, "resi": str(variant_pos)})
        else:
            # Zoom to the entire model if the specific position is missing
            local_view.zoomTo()
            display(HTML("<div style='border:1px solid #ffeeba; background:#fff3cd; color:#856404; padding:12px; margin:8px 0; border-radius:4px;'><b>Structure note:</b> Position 72 is outside the structural coordinates, showing the core domain overview instead.</div>"))
            
        local_view.setBackgroundColor("white")
        display(HTML(local_view._make_html()))


widgets.interact(
    explore_variant,
    selected_variant=widgets.Dropdown(options=variant_df["Variant"].tolist(), value="R248Q", description="Variant:"),
    window_size=widgets.IntSlider(value=6, min=3, max=12, step=1, description="Seq window:", continuous_update=False),
    show_structure=widgets.Checkbox(value=True, description="Show 3D structure")
);

print("Interactive explorer ready. Choose a variant to compare biochemical change with sequence and structural context.")

## 8. What We Can and Cannot Conclude

<div style="border:1px solid #f5c6cb; background:#f8d7da; color:#721c24; padding:16px; margin:8px 0; border-radius:4px;">
    <b>Important:</b> This notebook supports careful interpretation, not diagnosis. A missense change can look dramatic on paper and still have context-dependent behaviour. Likewise, a conservative-looking change can matter if it occurs in a sensitive structural environment.
</div>

It helps to separate three levels of statement:

- <b>Observation:</b> the variant changes one amino acid, sits in a particular region, and may lie close to DNA or zinc in a structure.
- <b>Interpretation:</b> this makes some molecular consequences more plausible, such as altered DNA contact, disturbed packing, or changed local stability.
- <b>Speculation:</b> claiming exact functional loss, clinical severity, or disease mechanism from this notebook alone.

A few limitations to keep in mind:

- We used a small teaching panel, not a comprehensive variant catalogue.
- Our biochemical shift score is a transparent heuristic, not a validated predictive model.
- A single structure captures one context and one conformation, not the full dynamic life of the protein.
- Real variant interpretation also needs experimental evidence, literature context, and often cell-level or organism-level data.

That caution is good science.

## Take-away

<div style="border:1px solid #b8daff; background:#d9edf7; color:#0c5460; padding:16px; margin:8px 0; border-radius:4px;">
    Congratulations! You have built a full variant-to-function interpretation workflow: you loaded a protein case study, parsed missense substitutions, mapped them onto sequence and domain context, added structural evidence, and compared their likely molecular consequences using careful and responsible language.
</div>

You have seen that:

- not all amino-acid substitutions are biochemically equivalent
- residue chemistry matters, but position matters too
- structural context can sharpen our interpretation dramatically
- computational context is useful for reasoning, but it is not proof

Good next steps would be to:

- add more TP53 variants and compare their contexts
- repeat the workflow for another protein with a strong domain structure
- combine this notebook with conservation analysis or literature-derived annotations
- compare multiple structures to see whether the local environment changes across states

Keep exploring — this is how molecular sequence changes start to become mechanistic questions.

![Noteable license](https://raw.githubusercontent.com/YusufNik/Noteable/refs/heads/main/images/Noteable%20Notebook%20Footer.png)